In [1]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

initialized the dmm connection and assign channel ch4
initialized the dmm connection and assign channel ch1
initialized the dmm connection and assign channel ch2
initialized the dmm connection and assign channel ch3
initialized the rigol dp821a connection and assign channel 1
initialized the rigol dp821a connection and assign channel 2
initialized the keysight e36232a connection
initialized the 2460 source meter
initialized the e-loader connection
initialized the scope connection
[relay_box <module>] initialize pcf8575 to be off state for all channels
[relay_box <module>] low channel 2 byte should be placed before high channel 2 byte for pcf8575


# Test A9

kt : ---> VDD3V3

ks : ---> dm.ch1 (I) ---> relay.ch1 (VIN)

ps.ch1 : ---> relay power

ps.ch2 : ---> relay.ch3 (EN)

dm.ch1 : ---> relay.ch1 (VIN)

dm.ch3 : ---> VIN-VOUT diff

relay.ch1 : VIN

relay.ch2 : terminal_1=VOUT, terminal_2=e-loader (10mA)

relay.ch3 : EN

loader : 10mA load

note : in some case, latch-up shows on IOUT. Recycle loader CC or CR, then retest.

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_9] VOUT_REG")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "VIN-VOUT (mV)", "I_VOUT (mA)", "DIODE_MODE_EN", "SW_STS", "RRG_STS"])

In [3]:
disable_all_ps()
relay.init_channels

v_vdd = 3.3
v_en = 3.3

v_start  = 5
v_finish = 30.1
step = 1
ndigit = 1

list_temp = list(np.arange(v_start, v_finish, step))
list_vin = [round(num, ndigit) for num in list_temp]

print(f"voltage step : {step}, round : {ndigit}")

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

dm.ch1.current_200mA

relay.ch1.enable # monitoring for vin
relay.ch2.enable # vout
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = 3.3, 0.1
ps.ch2.enable
delay(0.5)

relay.ch3.enable # en
delay(0.5)

# --------------------------------------------
ic.DIODE_MODE_EN = 1
ld.iset = 0.011
ld.enable

[relay_box <module>] disable all channels


channel            state
-----------------  -------
low channel  # 1   off
low channel  # 2   off
low channel  # 3   off
low channel  # 4   off
low channel  # 5   off
low channel  # 6   off
low channel  # 7   off
low channel  # 8   off
high channel # 9   off
high channel # 10  off
high channel # 11  off
high channel # 12  off
high channel # 13  off
high channel # 14  off
high channel # 15  off
high channel # 16  off


voltage step : 1, round : 1


relay channel 1 on : low 0xfe, high 0xff
relay channel 2 on : low 0xfc, high 0xff
relay channel 3 on : low 0xf8, high 0xff


In [ ]:
# ----------------------------------------------------------------------------

for vin in list_vin:
    
    ks.vset = vin
    
    vin_vout = round(dm.ch3.voltage_200mV * 1e+3, 6) # mV scale
    iout = round(dm.ch1.current_20mA, 6)
    reg_diode = ic.DIODE_MODE_EN
    sw_sts = ic.SW_STS
    rrg_sts = ic.RRG_STS
    
    ret = [vin, v_vdd, v_en, vin_vout, iout, reg_diode, sw_sts, rrg_sts]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

# debugging

In [ ]:
ic.write_byte(0xa0, 0xf9)
ic.write_byte(0xa0, 0x9f)

ic.DIODE_MODE_EN = 1

from time import time
ks.vset = 20
lap_start = time()

log.output_set_filename(f"[debugging] VOUT_REG, 100mA load, evb #1")
log.output_csv(["Time (s)", "Temperature", "VIN (V)", "VDD3V3 (V)", "EN (V)", "VIN-VOUT (mV)", "I_VOUT (mA)", "DIODE_MODE_EN", "SW_STS", "RRG_STS"])

while True:
    
    lap_time  = round(time() - lap_start, 1)
    m_temp    = round(dm.ch4.temperature,1)
    vin_vout  = round(dm.ch3.voltage_200mV * 1e+3, 3)
    iout      = round(dm.ch1.current_20mA * 1e+3, 3)
    reg_diode = ic.DIODE_MODE_EN
    sw_sts    = ic.SW_STS
    rrg_sts   = ic.RRG_STS
    tm_active = ic.read_byte(0xa0)
    reg_a9h   = ic.read_byte(0xa9)
    
    ret = [lap_time, m_temp, 20, 3.3, 3.3, vin_vout, iout, reg_diode, sw_sts, rrg_sts, tm_active, reg_a9h]
    log.output_csv(ret)
    
    print(ret)

In [2]:
disable_all_ps()